# TED Talks Transcript Extractor

TODO: Description

### 1. Dependencies

In [3]:
%pip install --quiet requests beautifulsoup4 pandas lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: C:\Users\estep\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import json
import re
import time
from typing import Optional

import pandas as pd
import requests
from bs4 import BeautifulSoup

### 2. Settings

In [ ]:
TARGET_LANGUAGES = ["zh", "en", "fi", "fr", "he", "ja", "pl"]
CHINESE_FALLBACKS = ["zh", "zh-cn", "zh-tw"]

SLUG_LIST = ["sir_ken_robinson_do_schools_kill_creativity"]

GRAPHQL_URL = "https://www.ted.com/graphql"

OUTPUT_PATH = "ted_talks_transcripts.tsv"

UA = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
)

HEADERS = {
    "User-Agent": UA,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

GRAPHQL_HEADERS = {
    "User-Agent": UA,
    "Accept": "application/json",
    "Content-Type": "application/json",
    "Origin": "https://www.ted.com",
    "Referer": "https://www.ted.com/",
}

session = requests.Session()
session.headers.update(HEADERS)

### 3. Fetching

In [10]:
def fetch_talk_metadata(slug: str) -> dict:
    url = f"https://www.ted.com/talks/{slug}"
    resp = session.get(url, timeout=30)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "lxml")
    tag = soup.find("script", id="__NEXT_DATA__")
    if tag is None or not tag.string:
        raise RuntimeError(f"__NEXT_DATA__ not found on {url}")

    data = json.loads(tag.string)
    page_props = data.get("props", {}).get("pageProps", {})

    video_data = page_props.get("videoData") or {}
    if not video_data and "video" in page_props:
        video_data = page_props["video"]

    player_data_raw = video_data.get("playerData")
    player_data = {}
    if isinstance(player_data_raw, str):
        try:
            player_data = json.loads(player_data_raw)
        except json.JSONDecodeError:
            player_data = {}
    elif isinstance(player_data_raw, dict):
        player_data = player_data_raw

    languages = []
    for entry in player_data.get("languages", []) or []:
        if isinstance(entry, dict):
            code = entry.get("languageCode") or entry.get("language_code")
            if code:
                languages.append(code)

    video_id = (
        str(video_data.get("id") or "")
        or str(player_data.get("id") or "")
        or str(player_data.get("videoId") or "")
    )

    speakers = []
    for s in (video_data.get("speakers") or []):
        if not s:
            continue
        if isinstance(s, str):
            speakers.append(s)
        elif isinstance(s, dict):
            name = (
                s.get("fullName")
                or f"{s.get('firstname', '')} {s.get('lastname', '')}".strip()
            )
            if name:
                speakers.append(name)

    video_type_raw = video_data.get("videoType")
    if isinstance(video_type_raw, dict):
        video_type = video_type_raw.get("name") or video_type_raw.get("title") or ""
    elif isinstance(video_type_raw, str):
        video_type = video_type_raw
    else:
        video_type = ""

    event_raw = video_data.get("eventName") or video_data.get("event") or ""
    if isinstance(event_raw, dict):
        event_raw = event_raw.get("name") or event_raw.get("title") or ""
    elif not isinstance(event_raw, str):
        event_raw = ""

    meta = {
        "slug": slug,
        "url": url,
        "video_id": video_id,
        "title": video_data.get("title") or player_data.get("title", ""),
        "description": video_data.get("description")
            or player_data.get("description", ""),
        "recorded_on": video_data.get("recordedOn")
            or player_data.get("recorded_at", ""),
        "published_at": video_data.get("publishedAt")
            or player_data.get("published_at", ""),
        "speakers": speakers,
        "available_languages": languages,
        "videoType": video_type,
        "event": event_raw,
    }
    return meta

In [11]:
SLUG = "sir_ken_robinson_do_schools_kill_creativity"

meta = fetch_talk_metadata(SLUG)
print(f"Title      : {meta['title']}")
print(f"Video id   : {meta['video_id']}")
print(f"Recorded   : {meta['recorded_on']}")
print(f"Published  : {meta['published_at']}")
print(f"# languages: {len(meta['available_languages'])}")
print("Description:")
print(meta["description"][:500], "..." if len(meta["description"]) > 500 else "")

missing = [
    lang
    for lang in TARGET_LANGUAGES
    if lang not in meta["available_languages"]
    and not (lang == "zh" and any(c in meta["available_languages"] for c in CHINESE_FALLBACKS))
]
print("Missing target languages:", missing or "none")

Title      : Do schools kill creativity?
Video id   : 66
Recorded   : 2006-02-25
Published  : 2006-06-27T00:11:00Z
# languages: 64
Description:
Sir Ken Robinson makes an entertaining and profoundly moving case for creating an education system that nurtures (rather than undermines) creativity. 
Missing target languages: none


#### 4. Extracting

In [ ]:
def _build_query(video_id: str, language: str) -> str:
    return (
        '{ translation(videoId:"' + video_id + '", language:"' + language + '") '
        '{ paragraphs { cues { text } } } }'
    )

def _graphql_transcript(video_id: str, language: str, timeout: int = 30) -> Optional[str]:
    payload = {"operationName": None, "query": _build_query(video_id, language)}
    resp = session.post(
        GRAPHQL_URL,
        json=payload,
        headers=GRAPHQL_HEADERS,
        timeout=timeout,
    )
    if resp.status_code != 200:
        print(f"  [{language}] HTTP {resp.status_code}")
        return None

    try:
        body = resp.json()
    except ValueError:
        print(f"  [{language}] invalid JSON response")
        return None

    translation = (body.get("data") or {}).get("translation")
    if not translation:
        return None

    lines = []
    for paragraph in translation.get("paragraphs") or []:
        cues = paragraph.get("cues") or []
        text = " ".join((c.get("text") or "").strip() for c in cues if c.get("text"))
        text = re.sub(r"\s+", " ", text).strip()
        if text:
            lines.append(text)
    return "\n".join(lines) if lines else None


def fetch_transcript(video_id: str, language: str, polite_delay: float = 0.5) -> Optional[str]:
    candidates = CHINESE_FALLBACKS if language == "zh" else [language]
    for lang_code in candidates:
        transcript = _graphql_transcript(video_id, lang_code)
        if transcript:
            time.sleep(polite_delay)
            return transcript
        time.sleep(polite_delay)
    return None

In [ ]:
transcripts = {}
for lang in TARGET_LANGUAGES:
    print(f"Fetching {lang} …")
    text = fetch_transcript(meta["video_id"], lang)
    transcripts[lang] = text
    if text:
        print(f"  ok — {len(text):,} chars")
    else:
        print(f"  not available")

Fetching zh …
  ok — 6,395 chars
Fetching en …
  ok — 17,574 chars
Fetching fi …
  ok — 14,827 chars
Fetching fr …
  ok — 17,112 chars
Fetching he …
  ok — 12,831 chars
Fetching ja …
  ok — 7,331 chars
Fetching pl …
  ok — 16,392 chars


#### 5. Saving

In [16]:
row = {
    "Ted talk title": meta["title"],
    "description": meta["description"],
    **{lang: transcripts.get(lang) for lang in TARGET_LANGUAGES},
}

df = pd.DataFrame([row])
core_cols = ["Ted talk title", "description", *TARGET_LANGUAGES]
extra_cols = [c for c in df.columns if c not in core_cols]
df = df[core_cols + extra_cols]

In [17]:
df

,Ted talk title,description,zh,en,fi,fr,he,ja,pl
0,Do schools kill creativity?,Sir Ken Robinson makes an entertaining and pro...,早上好，还好吗? 很好吧，对不对? 我已经飘飘然了! 我要飘走了。(笑声) 这次会议有三个主...,Good morning. How are you?\n(Audience) Good.\n...,Hyvää huomenta. Kuinka voitte? Olen ollut tava...,Bonjour. Comment ça va ? Public : Bien. Tout ç...,"בוקר טוב. מה שלומכם? היה נהדר, נכון? היה כיף א...",おはようございます。気分はいかがですか？素晴らしいですね、ここは すべてが驚嘆の連続です だ...,Dzień dobry. Jak się macie? Było super? Brak m...


In [ ]:
df.to_csv(OUTPUT_PATH, sep="\t", index=False)
print(f"Saved {len(df)} row(s) -> {OUTPUT_PATH}")

Saved 1 row(s) -> ted_talks_transcripts.tsv
